Задача: **Файн-тюнинг с LoRA**

Теория и пример: https://huggingface.co/learn/llm-course/chapter11/4

Текст задачи: Build on your fine-tuned model from the previous section, but fine-tune it with LoRA. Use the HuggingFaceTB/smoltalk dataset to fine-tune a deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B model, using the LoRA configuration we defined above.

Был проведён тест с deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B, но она слишко рассуждающая, задач реализована на модели
Qwen/Qwen2.5-1.5B-Instruct

Ниже приведен полный пример для **Google Colab (1 GPU)**. Он использует:

* Qwen/Qwen2.5-1.5B-Instruct
* HuggingFaceTB/smoltalk
* LoRA через PEFT
* TRL SFTTrainer

Структура решения:

1. Установка библиотек
2. Авторизация HF (при необходимости)
3. Загрузка модели
4. Настройка LoRA
5. Загрузка датасета
6. Подготовка датасета
7. Обучение LoRA
8. Сохранение адаптера
9. Загрузка базовой модели + LoRA
10. Тестирование и сравнение с базовой моделью

При написании и отладке кода использовался ChatGPT

## 1. Установка библиотек

In [ ]:
!pip -q install -U transformers datasets peft trl accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.3 MB/s eta 0:00:00


проблема несовместимости версий библиотек в текущем Colab.

У вас установился:

torchao==0.10.0

а последняя версия peft требует

torchao >= 0.16.0

(рекомендую) — удалить torchao

Он не нужен для обучения LoRA.

In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


##2. Авторизация HuggingFace (если модель требует доступа)

Для Qwen авторизация не нужна

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

KeyboardInterrupt: 

##3. Импорт библиотек и проверка GPU

In [ ]:
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
)

from trl import SFTTrainer

In [ ]:
# Проверка доступности GPU в Google Colab

import torch

print("=" * 50)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print(f"Total GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("GPU не обнаружен. Включите GPU:")
    print("Runtime -> Change runtime type -> Hardware accelerator -> GPU")
print("=" * 50)

CUDA available: True
GPU: Tesla T4
CUDA version: 12.8
Total GPU memory: 14.56 GB


##4. Загрузка модели

In [ ]:
#MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

##5. Конфигурация LoRA

(используйте вашу конфигурацию, если она была задана ранее)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

##6. Подключение LoRA

In [ ]:
model = get_peft_model(base_model, lora_config)

model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


##7. Загрузка датасета

In [ ]:
dataset = load_dataset(
    "HuggingFaceTB/smoltalk",
    "everyday-conversations"
)

dataset

README.md:   0%|          | 0.00/9.72k [00:00<?, ?B/s]

data/everyday-conversations/train-00000-(…): reconstructing file:   0%|          |  0.00B /  946kB            

data/everyday-conversations/train-00000-(…): downloading bytes:           |  0.00B            

data/everyday-conversations/test-00000-o(…): reconstructing file:   0%|          |  0.00B / 52.6kB            

data/everyday-conversations/test-00000-o(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2260 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/119 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['full_topic', 'messages'],
        num_rows: 2260
    })
    test: Dataset({
        features: ['full_topic', 'messages'],
        num_rows: 119
    })
})

##8. Подготовка датасета

SmolTalk хранит сообщения в виде списка диалогов.

In [ ]:
def format_example(example):

    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

    return {
        "text": text
    }


train_dataset = dataset["train"].map(format_example)

train_dataset = train_dataset.remove_columns(
    [c for c in train_dataset.column_names if c != "text"]
)

Map:   0%|          | 0/2260 [00:00<?, ? examples/s]

##9. Параметры обучения

In [18]:
training_args = TrainingArguments(

    output_dir="./lora_output",

    per_device_train_batch_size=2,

    gradient_accumulation_steps=8,

    max_steps=300, # увеличено после сильного совпадения ответов базовой модели и LoRA,
    #время обученя примерно 20 мин

    learning_rate=2e-4,

    num_train_epochs=1,

    logging_steps=20,

    save_steps=200,

    fp16=True,

    report_to="none",
)

##10. Создание Trainer

In [19]:
trainer = SFTTrainer(

    model=model,

    train_dataset=train_dataset,

    args=training_args,
)

##11. Обучение LoRA

Во время обучения вы увидите уменьшение loss.

In [20]:
trainer.train()

Step,Training Loss
20,0.437473
40,0.484228
60,0.506463
80,0.493272
100,0.495202
120,0.514650
140,0.515540


Step,Training Loss
20,0.437473
40,0.484228
60,0.506463
80,0.493272
100,0.495202
120,0.514650
140,0.515540
160,0.364487
180,0.339940
200,0.354101


TrainOutput(global_step=300, training_loss=0.41863351504007973, metrics={'train_runtime': 952.6958, 'train_samples_per_second': 5.038, 'train_steps_per_second': 0.315, 'total_flos': 8796059077761024.0, 'train_loss': 0.41863351504007973, 'epoch': 2.113274336283186})

##12. Сохранение LoRA

Будет сохранено примерно

deepseek_lora/

adapter_config.json

adapter_model.safetensors

tokenizer.json
...

Размер адаптера обычно около 10–30 МБ.

In [21]:
SAVE_PATH = "./deepseek_lora"

trainer.model.save_pretrained(SAVE_PATH)

tokenizer.save_pretrained(SAVE_PATH)

print("LoRA adapter saved.")

LoRA adapter saved.


##13. Загрузка базовой модели

In [22]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

##14. Загрузка LoRA

In [23]:
lora_model = PeftModel.from_pretrained(
    base_model,
    SAVE_PATH,
)

lora_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

## 15. Функция генерации

In [24]:
def generate(model, prompt):

    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
        )

    answer = tokenizer.decode(
        output[0],
        skip_special_tokens=True,
    )

    return answer

#16. Тестирование базовой модели

In [25]:
prompt = "Explain recursion like I am 10 years old."

print(generate(base_model, prompt))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Explain recursion like I am 10 years old.
assistant
Recursion is when you break down a big problem into smaller parts and solve those parts one at a time. It's like building with blocks - you take apart a block, look at its pieces, and then build it back up again.


##17. Тестирование LoRA

In [26]:
print(generate(lora_model, prompt))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Explain recursion like I am 10 years old.
assistant
Imagine you have a big box of toys. If someone asks you how many toys are inside, and they tell you to open the box and count the toys inside it, but then ask you again after opening the box once more, you're using recursion! You keep counting until you've opened the box as many times as asked.


##18. Сравнение нескольких запросов

In [27]:
prompts = [

    "Tell me a funny joke.",

    "Explain what AI is.",

    "How do airplanes fly?",

    "Write a short motivational message.",

    "My friend ignored me today. What should I do?",

    "I failed an exam. Can you encourage me?"

]

for p in prompts:

    print("=" * 80)
    print("PROMPT")
    print(p)

    print("\nBASE MODEL")
    print(generate(base_model, p))

    print("\nLORA MODEL")
    print(generate(lora_model, p))

PROMPT
Tell me a funny joke.

BASE MODEL
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Tell me a funny joke.
assistant
Why don't scientists trust atoms? Because they make up everything.

LORA MODEL
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Tell me a funny joke.
assistant
Why don't skeletons fight each other? Because they're brothers - from "The Hitchhiker's Guide to the Galaxy".
PROMPT
Explain what AI is.

BASE MODEL
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Explain what AI is.
assistant
AI stands for Artificial Intelligence. It's when machines and computer systems can perform tasks that usually require human intelligence, like learning, reasoning, and decision-making.

LORA MODEL
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Explain what AI is.
assistant
AI stands for Artificial Intelligence, which refers to the simulation of human intelli